# Part 4: Generation - Virtual Customer for Live Sales Practice

## Gym Sales Training Simulator - Conversational Practice Partner

Part 3 retrieves *past* conversations so a trainee can study what worked. Part 4 lets them
**practise live**: an LLM role-plays a gym customer drawn from our dataset's persona system,
and the trainee has to handle its objections in real time.

**Pipeline:** trainee describes a customer -> Part 3 recommender picks the closest archetype ->
persona + hidden objections build a system prompt -> Qwen2.5-7B-Instruct plays that customer ->
trainee practises -> a debrief judges each objection as resolved or not.

**Design principle carried over from the dataset:** the virtual customer holds *hidden
objections* it voices naturally when the topic comes up, and never announces them as a list.
This is exactly how Stage A of our data generation scripted the synthetic customers, so the
practice partner behaves like the customers in the training corpus.

**This part is evaluated, not just demonstrated.** Sections 6-7 audit the generated behaviour
(objection voicing, character discipline, turn length) and run a paired probe showing the
debrief distinguishes a strong sales approach from a weak one - the same philosophy as the
skill-signal validation we ran on the dataset itself.

**Where the model runs:** the Hugging Face Inference API, not inside the Space. A free Space is
CPU-only, where a 7B model needs ~30s per reply and risks OOM mid-demo. The API keeps the Part 5
Space light and interactive; every call has a fallback path so a transient failure degrades one
reply instead of crashing the app.

In [1]:
!pip install -q transformers accelerate huggingface_hub pandas sentence-transformers faiss-cpu
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 104.9 MB/s eta 0:00:00
NVIDIA A100-SXM4-40GB, 40960 MiB


## 1. Load the persona system from our dataset

The customer's identity comes from the data, not from hand-written descriptions:

- `gym_sales_dataset.jsonl` - the conversations (used by the recommender)
- `gym_sales_metadata.jsonl` - the **latent state** per conversation, including
  `hidden_objection_labels`

The metadata was never a feature in Part 3 - using it for retrieval would be leakage. Here it
plays a different, legitimate role: it is the *script* for the role-play, the same way it
scripted the synthetic customers during generation. Latent state generates behaviour; it still
never enters a search key or a model input on the prediction side.

In [2]:
import pandas as pd

BASE = "https://huggingface.co/datasets/elg4/Gym_Salesman_Dataset/resolve/main"

df = pd.read_json(f"{BASE}/gym_sales_dataset.jsonl", lines=True)
meta = pd.read_json(f"{BASE}/gym_sales_metadata.jsonl", lines=True)

# The metadata file predates the EDA dedup (12,000 rows vs 11,997) - align it
# to the published dataset by sample_id so both sides describe the same corpus.
meta = meta[meta["sample_id"].isin(set(df["sample_id"]))].reset_index(drop=True)

assert len(df) == len(meta) == 11997, f"mismatch: df={len(df)}, meta={len(meta)}"
assert set(df["sample_id"]) == set(meta["sample_id"]), "sample_id sets differ"
print(f"conversations: {len(df)} | metadata rows: {len(meta)} (aligned)")

conversations: 11997 | metadata rows: 11997 (aligned)


In [3]:
print("metadata columns:", list(meta.columns))
print("dataset columns :", list(df.columns))
print()
print(meta.iloc[0])

metadata columns: ['sample_id', 'skill_level', 'hidden_objection_ids', 'hidden_objection_labels', 'objections_resolved_flags', 'objections_resolved_count', 'persuasion_threshold', 'evaluator_reasoning']
dataset columns : ['sample_id', 'outcome', 'dialogue', 'decision_turn_index', 'persona', 'demographic', 'mood', 'gym', 'price', 'n_turns']

sample_id                                                                    0
skill_level                                                             novice
hidden_objection_ids                              [cancellation_maze, boredom]
hidden_objection_labels      [Every gym makes cancelling a nightmare - phon...
objections_resolved_flags                                        [False, True]
objections_resolved_count                                                    1
persuasion_threshold                                                         1
evaluator_reasoning          The advisor provided a general policy of cance...
Name: 0, dtype: object


In [4]:
# The metadata holds only latent state (skill, hidden objections, evaluator flags);
# the descriptive labels live in the dataset. Join on sample_id to build the catalogue.
meta_full = meta.merge(
    df[["sample_id", "persona", "mood", "demographic", "outcome"]],
    on="sample_id", how="left"
)
assert meta_full["persona"].notna().all(), "join left unmatched rows"

persona_catalogue = {}
for persona, grp in meta_full.groupby("persona"):
    objections = sorted({lab for labs in grp["hidden_objection_labels"] for lab in labs})
    persona_catalogue[persona] = {
        "objections": objections,
        "moods": sorted(grp["mood"].unique()),
        "demographics": sorted(grp["demographic"].dropna().unique()),
        "success_rate": (grp["outcome"] == "SUCCESS").mean(),
    }

for p, info in persona_catalogue.items():
    print(f"\n{p}   (corpus success rate {info['success_rate']:.1%})")
    for o in info["objections"]:
        print(f"    - {o}")


Ariel - Impulsive Commitment-Phobe   (corpus success rate 21.7%)
    - A year-long contract? I will quit in six weeks like I always do, and keep paying anyway.
    - Every gym makes cancelling a nightmare - phone calls, registered letters, retention agents.
    - Honestly, nothing will make me actually keep going after the first excitement wears off.
    - I get bored of the same routine within a month and stop showing up.

Noam - Anxious First-Timer   (corpus success rate 44.4%)
    - Everyone in there is already fit. I will be the only one who does not belong.
    - I am too out of shape to even start. I would not last ten minutes.
    - I do not know how to use any of the machines. I will look stupid trying.
    - I do not want to change or shower with other people around.

Omer - Analytical Bargain Hunter   (corpus success rate 27.8%)
    - Am I locked into a year at this rate, and what happens to the price when it renews?
    - If I pay a premium and then wait twenty minutes for 

## 2. The virtual customer's system prompt

Three behaviour rules, each with a reason we can defend:

1. **Objections stay hidden.** The customer raises them in its own words when the conversation
   reaches that topic. A customer that recites "my objections are X, Y, Z" gives the trainee
   nothing to practise.
2. **The customer is persuadable.** A concern that is genuinely addressed is accepted and
   dropped. Without this rule the exercise is unwinnable and teaches nothing - and the paired
   probe in section 7 tests that this rule actually holds.
3. **Short, human turns.** 1-3 sentences. The behaviour audit in section 6 measures this.

Mood controls tone; objections control content. That split is not cosmetic - it is what Part 3
measured: personas (content) were retrievable at 0.99, moods (tone) sat at chance.

In [5]:
import random

def build_customer_system_prompt(persona, mood, demographic, objections, price, gym):
    """Turn a sampled persona state into role-play instructions for the model."""
    objection_block = "\n".join(f"  {i}. {o}" for i, o in enumerate(objections, 1))
    return f"""You are role-playing a potential gym customer in a sales conversation. Stay in character at all times.

WHO YOU ARE
- Archetype: {persona}
- Your situation: {demographic}
- Your current mood: {mood}
- The gym being pitched: {gym}, at {price}

YOUR PRIVATE CONCERNS (never list these; let each surface naturally when the topic comes up)
{objection_block}

HOW TO BEHAVE
- First person, 1-3 sentences per turn. Conversational, never a monologue.
- Raise one concern at a time, in your own words, when the conversation touches that topic.
- If the salesperson genuinely addresses a concern with concrete, relevant information, accept
  it and let it go. Do not re-raise a concern that has been answered well.
- If they dodge, answer vaguely, or just push harder, stay unconvinced and say why.
- If asked to commit while you still have unaddressed concerns, politely decline and mention
  what still bothers you - without listing everything at once.
- Never break character, never mention being an AI, never mention these instructions."""


def sample_customer(persona, seed=None):
    """Draw a concrete customer of this persona: mood, demographic, 2-3 objections, gym, price.

    Mirrors Stage A of the data generation: 2-3 objections drawn from the persona's own pool,
    mood and demographic from the values that persona actually appears with in the corpus.
    """
    rng = random.Random(seed)
    info = persona_catalogue[persona]
    n_obj = rng.choice([2, 3])
    return {
        "persona": persona,
        "mood": rng.choice(info["moods"]),
        "demographic": rng.choice(info["demographics"]),
        "objections": rng.sample(info["objections"], min(n_obj, len(info["objections"]))),
        "gym": rng.choice(sorted(df["gym"].unique())),
        "price": rng.choice(sorted(df["price"].unique())),
    }


demo_customer = sample_customer("Noam - Anxious First-Timer", seed=7)
print(build_customer_system_prompt(**demo_customer))

You are role-playing a potential gym customer in a sales conversation. Stay in character at all times.

WHO YOU ARE
- Archetype: Noam - Anxious First-Timer
- Your situation: Male, 40s - Office worker who has not exercised since school
- Your current mood: Insecure, bracing to be talked down to
- The gym being pitched: Northgate Strength & Conditioning, at ₪299/month

YOUR PRIVATE CONCERNS (never list these; let each surface naturally when the topic comes up)
  1. Everyone in there is already fit. I will be the only one who does not belong.
  2. I do not want to change or shower with other people around.
  3. I do not know how to use any of the machines. I will look stupid trying.

HOW TO BEHAVE
- First person, 1-3 sentences per turn. Conversational, never a monologue.
- Raise one concern at a time, in your own words, when the conversation touches that topic.
- If the salesperson genuinely addresses a concern with concrete, relevant information, accept
  it and let it go. Do not re-rais

## 3. Calling the model

`Qwen/Qwen2.5-7B-Instruct` - the same family that generated and evaluated the dataset, so the
virtual customer speaks like the customers in the corpus.

Temperature 0.8 for the customer (varied across sessions - a deterministic customer would be
memorised by a trainee after two runs); temperature 0.0 for the debrief judge (judgement should
be reproducible).

In [6]:
# Local inference on the Colab GPU: no API quota, no credits, fully reproducible.
# Same model family that generated and evaluated the dataset, so the virtual customer
# speaks like the customers in the corpus.
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

CHAT_MODEL = "Qwen/Qwen2.5-7B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(CHAT_MODEL)
model_lm = AutoModelForCausalLM.from_pretrained(
    CHAT_MODEL,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
model_lm.eval()

print(f"{CHAT_MODEL} loaded on {model_lm.device}")
print(f"GPU memory allocated: {torch.cuda.memory_allocated() / 1024**3:.1f} GB")

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


... (2 outputs left out)

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Qwen/Qwen2.5-7B-Instruct loaded on cuda:0
GPU memory allocated: 14.2 GB


In [7]:
import time

FALLBACK_REPLY = "Sorry, could you say that again? I lost you for a second."


@torch.inference_mode()
def _chat(messages, max_new_tokens=120, temperature=0.8):
    """One chat completion on the local model. Returns the assistant's text."""
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([text], return_tensors="pt").to(model_lm.device)

    do_sample = temperature > 0
    out = model_lm.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=do_sample,
        temperature=temperature if do_sample else None,
        top_p=0.9 if do_sample else None,
        pad_token_id=tokenizer.eos_token_id,
    )
    generated = out[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()


def customer_reply(system_prompt, history, max_tokens=120, temperature=0.8):
    """history: [{"role": "user"|"assistant", "content": str}, ...]
    "user" = the salesperson (trainee), "assistant" = the virtual customer.

    Returns (reply_text, ok). The fallback path is kept from the API version: Part 5 may
    call a hosted model, and a live demo must degrade one reply rather than crash.
    """
    messages = [{"role": "system", "content": system_prompt}] + history
    try:
        return _chat(messages, max_new_tokens=max_tokens, temperature=temperature), True
    except Exception as e:
        print(f"[generation error] {type(e).__name__}: {e}")
        return FALLBACK_REPLY, False


# Smoke test + per-reply latency (the number a live demo depends on)
t0 = time.time()
reply, ok = customer_reply(build_customer_system_prompt(**demo_customer),
                           [{"role": "user", "content": "Hi! First time here?"}])
print(f"ok={ok} | {time.time() - t0:.1f}s | {reply}")

ok=True | 2.3s | Yes, first time. I haven't exercised much since school, so I'm a bit nervous. Is everyone already fit and in shape?


## 4. A scripted conversation

Fixed salesperson turns -> reproducible output for the walkthrough. The live version in Part 5
takes these turns from the trainee instead.

In [8]:
def run_scripted_conversation(customer, rep_turns, verbose=True):
    """Run a fixed salesperson script against the virtual customer.

    Returns (history, complete). `complete` is False if any turn fell back to canned text
    because the inference API failed - auditing or scoring such a transcript would measure
    our fallback string, not the model, so callers must check it.
    """
    system_prompt = build_customer_system_prompt(**customer)
    history, ok_count = [], 0

    if verbose:
        print("=" * 78)
        print(f"CUSTOMER: {customer['persona']}")
        print(f"  mood        : {customer['mood']}")
        print(f"  situation   : {customer['demographic']}")
        print(f"  gym / price : {customer['gym']} / {customer['price']}")
        print(f"  hidden concerns ({len(customer['objections'])}):")
        for o in customer["objections"]:
            print(f"    - {o}")
        print("=" * 78)

    for turn in rep_turns:
        history.append({"role": "user", "content": turn})
        reply, ok = customer_reply(system_prompt, history)
        ok_count += ok
        history.append({"role": "assistant", "content": reply})
        if verbose:
            print(f"\nSALES    : {turn}")
            print(f"CUSTOMER : {reply}")

    complete = ok_count == len(rep_turns)
    if verbose and not complete:
        print(f"\n[incomplete] only {ok_count}/{len(rep_turns)} turns came from the model")
    return history, complete


rep_script = [
    "Hi! Welcome in. Are you looking to join a gym, or just having a look around today?",
    "Totally get it. A lot of our members started exactly where you are. "
    "We run a free intro session with a trainer who walks you through every machine.",
    "That makes sense. There's no lock-in either - it's month to month, cancel any time online.",
    "So how are you feeling about it? Want me to book you in for that intro session?",
]

history, complete = run_scripted_conversation(demo_customer, rep_script)

CUSTOMER: Noam - Anxious First-Timer
  mood        : Insecure, bracing to be talked down to
  situation   : Male, 40s - Office worker who has not exercised since school
  gym / price : Northgate Strength & Conditioning / ₪299/month
  hidden concerns (3):
    - Everyone in there is already fit. I will be the only one who does not belong.
    - I do not want to change or shower with other people around.
    - I do not know how to use any of the machines. I will look stupid trying.

SALES    : Hi! Welcome in. Are you looking to join a gym, or just having a look around today?
CUSTOMER : Hi there! I'm just looking around today. I've been thinking about getting back into fitness, but I’m not sure if I should commit right away.

SALES    : Totally get it. A lot of our members started exactly where you are. We run a free intro session with a trainer who walks you through every machine.
CUSTOMER : That sounds helpful. I’m a bit nervous about using the machines though. I don’t want to look stupi

## 5. Debrief: did the trainee resolve the objections?

Practice without feedback teaches little. We reuse the **Stage C evaluator protocol** from the
data generation: judge each hidden objection separately, resolved or not, at temperature 0.

**Known limitation, measured, disclosed.** Our EDA hand-labelled 46 of these judgements:
Cohen's kappa = 0.217, with 20 of 21 disagreements in one direction - the evaluator is
consistently *stricter* than a human. A calibration gap, not random noise. So the debrief is
presented as conservative coaching feedback, not as a score; "not fully resolved" from a judge
that errs strict is the safe direction for a training tool.

In [9]:
import json as _json


def _extract_json(text):
    """Tolerant JSON extraction: strip fences, then take the first balanced {...} block.

    The model sometimes appends commentary after the JSON, so a greedy regex would
    swallow it. Counting braces returns exactly the first complete object, and the
    in-string tracking keeps braces inside a "why" sentence from breaking the count.
    """
    text = text.replace("```json", "").replace("```", "").strip()
    start = text.find("{")
    if start == -1:
        raise ValueError(f"no JSON object in: {text[:200]!r}")

    depth, in_str, esc = 0, False, False
    for i, ch in enumerate(text[start:], start):
        if in_str:
            if esc:
                esc = False
            elif ch == "\\":
                esc = True
            elif ch == '"':
                in_str = False
        elif ch == '"':
            in_str = True
        elif ch == "{":
            depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0:
                return _json.loads(text[start:i + 1])
    raise ValueError(f"unbalanced JSON in: {text[:200]!r}")


def debrief(customer, history):
    """Judge each hidden objection separately - same protocol as Stage C of generation.

    The prompt demands one verdict per concern, in order: an early run returned a single
    judgement and silently dropped two concerns, which still printed a plausible-looking
    score. Coverage is enforced here and verified again in print_debrief.
    """
    transcript = "\n".join(
        f"{'SALES' if m['role'] == 'user' else 'CUSTOMER'}: {m['content']}" for m in history
    )
    n_obj = len(customer["objections"])
    obj_list = "\n".join(f"{i}. {o}" for i, o in enumerate(customer["objections"], 1))

    prompt = f"""Read this gym sales conversation and judge each of the customer's concerns.

CONVERSATION:
{transcript}

THE CUSTOMER'S CONCERNS:
{obj_list}

For each concern, decide whether the salesperson resolved it: did they address it directly with
concrete, relevant information, and did the customer accept that answer? A concern that was
ignored, deflected, or only partially answered is NOT resolved. A concern the customer never
raised was not resolved either.

Respond with JSON only, no other text. Return exactly {n_obj} judgements - one for each concern
above, numbered 1 to {n_obj}, in order. Do not omit any concern.
{{"judgements": [{{"concern": 1, "resolved": true, "why": "<one short sentence>"}}]}}"""

    raw = None
    try:
        raw = _chat([{"role": "user", "content": prompt}], max_new_tokens=400, temperature=0.0)
        return _extract_json(raw)["judgements"]
    except Exception as e:
        print(f"[debrief failed] {type(e).__name__}: {e}")
        if raw is not None:
            print(f"[raw output] {raw[:300]}")
        return []


def print_debrief(customer, judgements):
    """Print one line per concern. Concerns the judge skipped are shown as NOT JUDGED
    rather than silently folded into the denominator."""
    n_obj = len(customer["objections"])
    by_idx = {}
    for j in judgements:
        idx = j.get("concern", 0) - 1
        if 0 <= idx < n_obj and idx not in by_idx:
            by_idx[idx] = j

    print("=" * 78)
    print("DEBRIEF")
    print("=" * 78)

    resolved = 0
    for idx, concern in enumerate(customer["objections"]):
        j = by_idx.get(idx)
        if j is None:
            print(f"[NOT JUDGED  ] {concern}")
            print("               judge returned no verdict for this concern")
            continue
        resolved += bool(j.get("resolved"))
        mark = "RESOLVED    " if j.get("resolved") else "NOT RESOLVED"
        print(f"[{mark}] {concern}")
        print(f"               {j.get('why', '')}")

    judged = len(by_idx)
    base = persona_catalogue[customer["persona"]]["success_rate"]
    print(f"\nResolved {resolved}/{n_obj} concerns"
          + (f"  ({n_obj - judged} not judged)" if judged < n_obj else ""))
    print(f"In the training corpus, this customer type closes {base:.1%} of the time "
          f"(dataset overall: {(df['outcome'] == 'SUCCESS').mean():.1%}).")
    return resolved


judgements = debrief(demo_customer, history)
_ = print_debrief(demo_customer, judgements)

DEBRIEF
[NOT RESOLVED] Everyone in there is already fit. I will be the only one who does not belong.
               The salesperson did not address this concern.
[NOT RESOLVED] I do not want to change or shower with other people around.
               The salesperson did not address this concern.
[RESOLVED    ] I do not know how to use any of the machines. I will look stupid trying.
               The salesperson addressed the concern by offering an intro session with a trainer.

Resolved 1/3 concerns
In the training corpus, this customer type closes 44.4% of the time (dataset overall: 27.4%).


## 6. Behaviour audit: does the generated customer actually behave as designed?

A role-play prompt is a claim, not a guarantee. We audit generated conversations the same way
we audited the dataset itself, with three checks per conversation:

- **Objection voicing (keyword floor).** Does the customer's text share content words with its
  hidden objection labels? Same method and same caveat as the dataset EDA, where this floor was
  79.7%: keyword overlap *understates* true voicing because a customer can express "fear of
  looking clueless" without using either word. A floor, reported as a floor.
- **Character discipline.** The text must never contain instruction-leak markers
  ("AI", "role-play", "instructions", "language model").
- **Turn length.** Mean words per customer turn should be conversational (< 60), per the
  1-3 sentence rule.

Six customers (all four personas represented), three neutral rep turns each - neutral, so the
customer has room to bring its own concerns up.

In [10]:
STOPWORDS = {"the", "a", "an", "of", "to", "and", "or", "in", "on", "for", "with",
             "is", "are", "be", "being", "about", "that", "this", "it", "its", "as",
             "at", "by", "from", "will", "would", "might", "may", "can", "could",
             "not", "no", "too", "very", "than", "then", "there", "here", "have",
             "has", "had", "do", "does", "did", "so", "if", "but", "they", "he",
             "she", "we", "you", "i", "my", "your", "their", "his", "her", "our"}

LEAK_MARKERS = ["as an ai", "language model", "role-play", "roleplay", "my instructions",
                "system prompt", "i am an assistant"]

NEUTRAL_SCRIPT = [
    "Hi! Welcome in. What brings you to the gym today?",
    "Good to know. Is there anything you'd want to ask me about the place before we go on?",
    "Fair enough. Anything else on your mind about joining?",
]


def audit_conversation(customer, history):
    cust_turns = [m["content"] for m in history if m["role"] == "assistant"]
    cust_text = " ".join(cust_turns).lower()

    voiced = 0
    for obj in customer["objections"]:
        keywords = {w.strip(".,!?'\"") for w in obj.lower().split()} - STOPWORDS
        if any(k in cust_text for k in keywords if len(k) > 3):
            voiced += 1

    leaks = [m for m in LEAK_MARKERS if m in cust_text]
    mean_words = sum(len(t.split()) for t in cust_turns) / max(len(cust_turns), 1)
    return {"voiced": voiced, "n_obj": len(customer["objections"]),
            "leaks": leaks, "mean_words": mean_words}


audit_customers = [sample_customer(p, seed=s)
                   for s, p in enumerate([
                       "Noam - Anxious First-Timer",
                       "Yuval - Procrastinating CS Student",
                       "Omer - Analytical Bargain Hunter",
                       "Ariel - Impulsive Commitment-Phobe",
                       "Noam - Anxious First-Timer",
                       "Omer - Analytical Bargain Hunter",
                   ])]

audit_rows, skipped = [], 0
for cust in audit_customers:
    hist, ok = run_scripted_conversation(cust, NEUTRAL_SCRIPT, verbose=False)
    if not ok:
        skipped += 1          # API failure: the transcript is our fallback string, not the model
        continue
    audit_rows.append({"persona": cust["persona"].split(" - ")[0],
                       **audit_conversation(cust, hist)})

if skipped:
    print(f"[warning] {skipped}/{len(audit_customers)} conversations excluded - the inference "
          f"API did not complete them. Auditing fallback text would measure our error handler, "
          f"not the model. Re-run when the API is available.\n")

if not audit_rows:
    print("No complete conversations to audit.")
else:
    audit_df = pd.DataFrame(audit_rows)
    audit_df["voicing_rate"] = audit_df["voiced"] / audit_df["n_obj"]
    print(audit_df[["persona", "voiced", "n_obj", "voicing_rate", "leaks", "mean_words"]])

    v, n = audit_df["voiced"].sum(), audit_df["n_obj"].sum()
    print(f"\nConversations audited   : {len(audit_df)}/{len(audit_customers)}")
    print(f"Objection voicing floor : {v}/{n} = {v / n:.1%} "
          f"(dataset EDA floor for comparison: 79.7%)")
    print(f"Instruction leaks       : {sum(bool(l) for l in audit_df['leaks'])} conversations")
    print(f"Mean words per turn     : {audit_df['mean_words'].mean():.1f} (target < 60)")

  persona  voiced  n_obj  voicing_rate leaks  mean_words
0    Noam       3      3           1.0    []   29.000000
1   Yuval       2      2           1.0    []   28.333333
2    Omer       2      2           1.0    []   35.000000
3   Ariel       1      2           0.5    []   26.000000
4    Noam       2      2           1.0    []   22.666667
5    Omer       3      3           1.0    []   39.333333

Conversations audited   : 6/6
Objection voicing floor : 13/14 = 92.9% (dataset EDA floor for comparison: 79.7%)
Instruction leaks       : 0 conversations
Mean words per turn     : 30.1 (target < 60)


## 7. Paired probe: does the debrief reward good selling?

The persuadability rule (section 2) is testable: the **same customer** (same seed, same hidden
objections) faces two different salespeople -

- **Weak rep:** generic pushing, never addresses a concern.
- **Strong rep:** concrete answers aimed at this customer's actual concern types.

If the simulator and debrief work, the strong rep resolves more objections. This is the Part 4
analogue of the dataset's skill-signal validation (novice 22.8% -> expert 30.9% success): the
system must respond to *what the seller does*, or it trains nothing.

One paired run is a directional probe, not a statistical claim - the dataset-level version of
this test is the 12,000-conversation skill gradient we already published.

In [12]:
probe_customer = sample_customer("Noam - Anxious First-Timer", seed=11)

weak_script = [
    "Hey, looking to join? This is the best gym in the city, honestly.",
    "You won't find a better deal anywhere, trust me. Everyone loves it here.",
    "So should I sign you up? The offer might not last.",
]

strong_script = [
    "Hi! Welcome. First time in a gym, or coming back after a break?",
    "That's really common, and honestly the first step is the hardest part. We do a free "
    "intro session where a trainer shows you every machine one-on-one, at your pace - "
    "no crowd, no pressure.",
    "And so you know, membership is month to month with online cancellation, and the first "
    "month is refundable. If it turns out not to be for you, you lose nothing. "
    "Would trying the intro session feel manageable?",
]

print("### WEAK REP ###")
weak_hist, weak_ok = run_scripted_conversation(probe_customer, weak_script)
print("\n\n### STRONG REP ###")
strong_hist, strong_ok = run_scripted_conversation(probe_customer, strong_script)

if not (weak_ok and strong_ok):
    print("\n[probe aborted] at least one conversation did not complete; "
          "comparing them would compare fallback text.")
else:
    print("\n")
    weak_resolved = print_debrief(probe_customer, debrief(probe_customer, weak_hist))
    print("\n")
    strong_resolved = print_debrief(probe_customer, debrief(probe_customer, strong_hist))

    print("\n" + "=" * 78)
    print(f"PAIRED RESULT: weak rep resolved {weak_resolved}, "
          f"strong rep resolved {strong_resolved} "
          f"(same customer, same hidden concerns)")

### WEAK REP ###
CUSTOMER: Noam - Anxious First-Timer
  mood        : Soft-spoken, apologising for taking up time
  situation   : Male, 40s - Office worker who has not exercised since school
  gym / price : Ironwood Fitness / ₪299/month
  hidden concerns (3):
    - I do not want to change or shower with other people around.
    - I do not know how to use any of the machines. I will look stupid trying.
    - Everyone in there is already fit. I will be the only one who does not belong.

SALES    : Hey, looking to join? This is the best gym in the city, honestly.
CUSTOMER : Hi, thanks for offering that. I've actually been thinking about getting back into exercise, but I'm a bit nervous about starting at a new gym.

SALES    : You won't find a better deal anywhere, trust me. Everyone loves it here.
CUSTOMER : That sounds great, but I'm really worried about feeling out of place. I mean, everyone seems so fit, and I don't want to look silly.

SALES    : So should I sign you up? The offer mig

## 8. Connecting Part 3 to Part 4

The trainee does not need to know our archetype names. They describe a customer in free text,
the Part 3 recommender retrieves the closest conversations, and the archetype of those
conversations becomes the practice partner. This loads the **published embeddings** - no
re-encoding, and the exact retrieval path the Part 5 app runs.

**Majority vote over 5 neighbours, not top-1.** Part 3's own output shows why: the "tough
negotiator" query returned one Yuval above two Omers. In Part 3 a single miss costs one
less-relevant result; here it would aim the entire practice session at the wrong customer.
Voting absorbs exactly that failure mode.

In [13]:
import pickle, faiss, numpy as np
from huggingface_hub import hf_hub_download
from sentence_transformers import SentenceTransformer

pkl_path = hf_hub_download(repo_id="elg4/Gym_Salesman_Dataset",
                           filename="embeddings/gym_conversation_embeddings.pkl",
                           repo_type="dataset")
with open(pkl_path, "rb") as f:
    store = pickle.load(f)

embed_model = SentenceTransformer(store["model_name"])
query_prefix = store["query_prefix"]
embeddings = store["embeddings"]

index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)

# Align pickle rows to dataframe rows via sample_id - never assume ordering
id_to_row = {sid: i for i, sid in enumerate(df["sample_id"].to_numpy())}
row_order = np.array([id_to_row[s] for s in store["sample_ids"]])

print(f"loaded {embeddings.shape[0]} vectors | model: {store['model_name']}")


def persona_from_query(query, top_k=5):
    """Free-text description -> archetype to practise against (majority vote over top_k)."""
    q = embed_model.encode([query_prefix + query], convert_to_numpy=True,
                           normalize_embeddings=True)
    _, idx = index.search(q, top_k)
    hits = df.iloc[row_order[idx[0]]]
    return hits["persona"].mode()[0]


for q in [
    "someone who is scared of gyms and has never worked out before",
    "a guy who keeps saying he will start next month",
    "a customer who argues about every shekel",
    "a tough negotiator who thinks the membership is overpriced",
]:
    print(f'"{q}"\n   -> {persona_from_query(q)}\n')

embeddings/gym_conversation_embeddings.p(…): reconstructing file:   0%|          |  0.00B / 18.5MB            

embeddings/gym_conversation_embeddings.p(…): downloading bytes:           |  0.00B            

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

... (6 outputs left out)

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

loaded 11997 vectors | model: BAAI/bge-small-en-v1.5
"someone who is scared of gyms and has never worked out before"
   -> Noam - Anxious First-Timer

"a guy who keeps saying he will start next month"
   -> Ariel - Impulsive Commitment-Phobe

"a customer who argues about every shekel"
   -> Omer - Analytical Bargain Hunter

"a tough negotiator who thinks the membership is overpriced"
   -> Omer - Analytical Bargain Hunter



In [14]:
# End-to-end: free-text request -> matched archetype -> live role-play -> debrief
trainee_request = "I want to practise on someone who thinks everything is too expensive"

matched_persona = persona_from_query(trainee_request)
practice_customer = sample_customer(matched_persona, seed=3)

print(f'REQUEST : "{trainee_request}"')
print(f"MATCHED : {matched_persona}\n")

e2e_hist = run_scripted_conversation(practice_customer, [
    "Hi there! Looking to get started with a membership?",
    "I hear you on the price. Let me break down what's included - full equipment access, "
    "all classes, and a personal program review every month - so you can judge the value.",
    "And compared to the two other gyms on this street we're 30 shekels cheaper with more "
    "included. If it helps, the first month is fully refundable, so trying it costs nothing.",
])
_ = print_debrief(practice_customer, debrief(practice_customer, e2e_hist))

REQUEST : "I want to practise on someone who thinks everything is too expensive"
MATCHED : Yuval - Procrastinating CS Student

CUSTOMER: Yuval - Procrastinating CS Student
  mood        : Tired but genuinely hoping this time is different
  situation   : Male, late 20s - Junior developer working long unpredictable sprint hours
  gym / price : The Forge Gym / ₪259/month
  hidden concerns (2):
    - Even if I go, I would not know what to do inside. I will wander around and waste the hour.
    - I will end up paying for months I never use, like every other subscription I have.

SALES    : Hi there! Looking to get started with a membership?
CUSTOMER : Hey, thanks for reaching out! Yeah, I’ve been thinking about joining a gym to help keep me on track with my health goals. What makes The Forge Gym stand out from others?

SALES    : I hear you on the price. Let me break down what's included - full equipment access, all classes, and a personal program review every month - so you can judge the v

TypeError: list indices must be integers or slices, not str

## Notes

- **The virtual customer is built from the dataset's own persona system** - personas, moods,
  demographics and `hidden_objection_labels` are read from the HF repo at runtime, so the
  practice partner and the training corpus cannot drift apart. The metadata that was off-limits
  as a retrieval feature in Part 3 (leakage) is legitimately the role-play script here, exactly
  as it was in Stage A of generation.
- **Generation is audited, not assumed** (section 6): objection-voicing keyword floor
  (methodology identical to the dataset EDA's 79.7% floor), zero-tolerance instruction-leak
  check, and turn-length check.
- **The feedback loop is probed** (section 7): the same customer against a weak and a strong
  rep, expecting the debrief to separate them - the Part 4 analogue of the dataset's
  novice-to-expert skill gradient.
- **The debrief reuses the Stage C evaluator protocol** and inherits its measured limitation:
  kappa = 0.217 vs human labels, one-directional (stricter). Disclosed and framed as
  conservative coaching feedback, not a score.
- **Inference on the HF Inference API** keeps the Part 5 Space light; every call degrades
  gracefully on failure. Customer at temperature 0.8 (variety), judge at 0.0 (reproducibility).
- **Archetype selection by majority vote over 5 neighbours** absorbs single-retrieval misses
  that Part 3's own sanity check surfaced.